# REGPLEX_Local — REGPLEX v13

Training-free perplexity valley detector.
Use this notebook to run and explore REGPLEX v13 locally.


## Scientific Hypothesis

> "Identify extended low-perplexity genomic valleys relative to their local genomic background using a training-free information-theoretic framework."

REGPLEX v13 is **not** a promoter predictor, not a motif predictor, and not a classifier.
No training data. No species-specific parameters. No machine learning.


## Pipeline

```
DNA → Dinucleotide Perplexity (window=17) → Savitzky-Golay → PDS → Bounded Kadane → Expand → Merge → Score → Motifs
```


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from regplex_core import (
    parse_fasta,
    compute_di_perplexity,
    smooth_perplexity,
    compute_pds,
    analyze_sequence,
    domains_dataframe,
    export_table,
)
from motif_engine import compile_motifs, annotate_domains
from visualization import (
    plot_perplexity_landscape,
    plot_smoothed_perplexity,
    plot_three_window,
    plot_pds_landscape,
    plot_valley_ranking,
    plot_motif_architecture,
    plot_algorithm_workflow,
)


## 2. Load FASTA

In [ ]:
with open('examples/ecoli.fasta', encoding='utf-8') as fh:
    fasta_text = fh.read()

records = parse_fasta(fasta_text)
print(f'Loaded {len(records)} sequence(s)')
for header, seq in records:
    print(f'  {header}: {len(seq)} bp')

header, seq = records[0]


## 3. Dinucleotide Perplexity (window = 17)

In [ ]:
di = compute_di_perplexity(seq, window=17)
print(f'Perplexity profile: {di.shape[0]} positions')
print(f'Mean: {np.nanmean(di):.3f}  Min: {np.nanmin(di):.3f}  Max: {np.nanmax(di):.3f}')


## 4. Savitzky-Golay Smoothing (applied once)

In [ ]:
smoothed = smooth_perplexity(di, window_length=21, poly_order=3)
print(f'Smoothed profile: {smoothed.shape[0]} positions')


## 5. Perplexity Depression Score (PDS)

In [ ]:
pds = compute_pds(smoothed, flank_size=100, spacer_size=50)
print(f'PDS profile: {pds.shape[0]} positions')
print(f'Positive PDS fraction: {np.nanmean(pds > 0):.3f}')
print(f'PDS range: [{np.nanmin(pds):.3f}, {np.nanmax(pds):.3f}]')


## 6. Full Pipeline (one call)

In [ ]:
result = analyze_sequence(header, seq)

print(f'Valleys detected: {len(result.domains)}')
for d in result.domains:
    print(f'  {d["ID"]}  {d["Start"]}-{d["End"]}  {d["Length"]} bp  '
          f'PDSMean={d["PDSMean"]:.3f}  Score={d["ValleyScore"]:.4f}')


## 7. Motif Annotation

In [ ]:
motifs = compile_motifs('TATAWAWR\nGGGN{1,7}GGG')
annotate_domains(result.domains, motifs)
for d in result.domains:
    print(f'{d["ID"]}: {d["MotifCount"]} motifs — {d["Motifs"]}')


## 8. Visualizations

In [ ]:
# Raw perplexity
fig = plot_perplexity_landscape(result.di, result.domains)
fig.show()


In [ ]:
# Smoothed perplexity with valleys
fig = plot_smoothed_perplexity(result.di, result.smoothed_di, result.domains)
fig.show()


In [ ]:
# PDS landscape
fig = plot_pds_landscape(result.pds, result.domains)
fig.show()


In [ ]:
# Three-window illustration (first valley)
if result.domains:
    fig = plot_three_window(
        result.smoothed_di, result.pds, result.domains[0],
        flank_size=100, spacer_size=50,
    )
    fig.show()


In [ ]:
# Valley ranking
fig = plot_valley_ranking(result.domains)
fig.show()


## 9. Export Results

In [ ]:
df = domains_dataframe([result])
df.to_csv('examples/sample_output_v13.csv', index=False)
print(f'Exported {len(df)} valleys to examples/sample_output_v13.csv')
df[['ID', 'Start', 'End', 'Length', 'PDSMean', 'Persistence', 'ValleyScore', 'Rank']]
